In [1]:
# !pip install datasets
# !pip install googletrans

### Dataset loading and preparation

In [2]:
from datasets import load_dataset

ds = load_dataset("facebook/xnli", "en")

m:\python_projects\AlignScore\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ds

DatasetDict({
    train: Dataset({
        features: ['premise', 'hypothesis', 'label'],
        num_rows: 392702
    })
    test: Dataset({
        features: ['premise', 'hypothesis', 'label'],
        num_rows: 5010
    })
    validation: Dataset({
        features: ['premise', 'hypothesis', 'label'],
        num_rows: 2490
    })
})

In [4]:
import random
random.seed(42)

ds_subset = ds['train'].shuffle(seed=42).select(range(10010))

In [5]:
ds_subset

Dataset({
    features: ['premise', 'hypothesis', 'label'],
    num_rows: 10010
})

In [6]:
ds_subset[:5]

{'premise': ["I 'll hurry over that part .",
  'Shall I tell you why you have been so vehement against Mr. Inglethorp ?',
  'well you know that brings up the interesting subject too you know what would you have who who who would determine what these people do',
  'A great Sather made the sun remain in one place too long , and the heat became too great .',
  'Of course , it will be generally known to-morrow . " John reflected .'],
 'hypothesis': ['" I \'ll be quick with that part . "',
  "I can tell you why you 're being so vehement against Mr. Inglethorp .",
  'It begs the question of who gets to say what the other people do .',
  'It got too hot when a Sather kept the sun in one spot .',
  'The news was about to break , and John had announced that he found out the newspaper would be announcing it tomorrow to the public .'],
 'label': [0, 0, 0, 0, 1]}

### Translation functions

In [7]:
from googletrans import Translator

translator = Translator()

In [8]:
async def translate_text(text, translator, src='en', dest='ru'):
    try:
        src = await translator.detect(text)
        translation = await translator.translate(text, src=src.lang, dest=dest)
        return str(translation.text)
    except Exception as e:
        print(f"Error translating text starting with '{text[:30]}...': {e}")
        return None

In [9]:
translation = await translate_text("i'm sorry, my cat is yelling all the time", translator)
print(translation)

Извините, моя кошка все время кричит


In [10]:
import asyncio

async def translate_text_batch(texts, translator, src='en', dest='ru'):
    tasks = [asyncio.create_task(translate_text(text, translator, src, dest)) for text in texts]
    results = await asyncio.gather(*tasks, return_exceptions=True)
    
    cleaned_results = []
    for idx, result in enumerate(results):
        if isinstance(result, Exception):
            print(f"Exception occurred for text '{texts[idx][:30]}...': {result}")
            cleaned_results.append(texts[idx])
        else:
            cleaned_results.append(result)
    return cleaned_results

In [11]:
texts = [ds_subset[0]['premise'], ds_subset[0]['hypothesis']]

In [12]:
print(await translate_text_batch(texts=texts, translator=translator, src='en', dest='ru'))

['Я спешу над этой частью.', '«Я буду быстрым с этой частью».']


### Translation loop

In [13]:
translated_premises = []
original_premises = []
translated_hypotheses = []
original_hypotheses = []
saved_labels = []
n_rows = 0

batch_size = 32
n = len(ds_subset)

In [14]:
from tqdm import tqdm

for i in tqdm(range(0, n, batch_size), desc="Translating"):
    batch = ds_subset[i:i+batch_size]
    premises = batch['premise']
    hypotheses = batch['hypothesis']
    labels = batch['label']
    
    premise_translations = await translate_text_batch(premises, translator)
    hypothesis_translations = await translate_text_batch(hypotheses, translator)
    
    for p, h, p_trans, h_trans, l in zip(premises, hypotheses, premise_translations, hypothesis_translations, labels):
        if n_rows < 10000 and p_trans is not None and h_trans is not None:
            original_premises.append(p)
            original_hypotheses.append(h)
            translated_premises.append(p_trans)
            translated_hypotheses.append(h_trans)
            saved_labels.append(l)
            n_rows += 1
        else:
            print("Skipping pair due to translation error.")

Translating:  42%|████▏     | 130/313 [01:01<01:28,  2.06it/s]

Error translating text starting with 'Don Cazar decides , Bartolomé ...': invalid source language


Translating:  42%|████▏     | 131/313 [01:01<01:25,  2.13it/s]

Skipping pair due to translation error.


Translating: 100%|██████████| 313/313 [02:28<00:00,  2.11it/s]

Skipping pair due to translation error.
Skipping pair due to translation error.
Skipping pair due to translation error.
Skipping pair due to translation error.
Skipping pair due to translation error.
Skipping pair due to translation error.
Skipping pair due to translation error.
Skipping pair due to translation error.
Skipping pair due to translation error.


### Save dataset

In [16]:
assert len(translated_premises) == len(translated_hypotheses) == len(saved_labels) == len(original_premises) == len(original_hypotheses)

In [17]:
from datasets import Dataset

ru_split = Dataset.from_dict({
    "premise": translated_premises,
    "hypothesis": translated_hypotheses,
    "label": saved_labels,
})

en_split = Dataset.from_dict({
    "premise": original_premises,
    "hypothesis": original_hypotheses,
    "label": saved_labels,
})

In [18]:
ru_split

Dataset({
    features: ['premise', 'hypothesis', 'label'],
    num_rows: 10000
})

In [19]:
from datasets import DatasetDict

final_dataset = DatasetDict({
    "ru": ru_split,
    "en": en_split,
})

In [20]:
final_dataset

DatasetDict({
    ru: Dataset({
        features: ['premise', 'hypothesis', 'label'],
        num_rows: 10000
    })
    en: Dataset({
        features: ['premise', 'hypothesis', 'label'],
        num_rows: 10000
    })
})

In [21]:
final_dataset.push_to_hub("MilyaShams/xnli_ru_en_10k", private=False)

Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.19s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/MilyaShams/xnli_ru_en_10k/commit/740d86a6d4af1cf4867f3b5adca134a1bf2f19ce', commit_message='Upload dataset', commit_description='', oid='740d86a6d4af1cf4867f3b5adca134a1bf2f19ce', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/MilyaShams/xnli_ru_en_10k', endpoint='https://huggingface.co', repo_type='dataset', repo_id='MilyaShams/xnli_ru_en_10k'), pr_revision=None, pr_num=None)